# Run and plot a `LAPDSim1D` simulation

This notebook:
1. Builds a `LAPDSim1D` from `default_config()`, applies your **parameter/flag overrides**, runs it, and saves an HDF5 result.
2. Renders the app-style contour and summary plots inline.
3. Shows main-discharge time-slice profiles at **15 ms and 19 ms** only.

All z-axis plots include the vertical dashed **port markers** (ports 20, 29, 40) used by `bapsf_app`.

Run the notebook from `cablp/scripts/` (it imports the CLI helper `plot_sim1d_run.py` from that directory).

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from cablp.solvers._sim1d import (
    LAPDSim1D,
    ProgressPrinter1D,
    default_config,
    load_result_hdf5,
    summarize_result,
)

# The CLI plotting helpers live next to this notebook in cablp/scripts/.
# Importing the module sets the Agg backend, so re-assert the inline backend after.
sys.path.insert(0, str(Path.cwd()))
import plot_sim1d_run as psr
%matplotlib inline

## Configuration

Edit `param_overrides` / `flag_overrides` to override the defaults. Anything left commented uses the value from `default_config()`. Call `default_config()` in a scratch cell to see every available key.

The run controls below map to `sim.start_simulation(...)`; leave them `None` to use the config defaults.

In [ ]:
# --- Parameter overrides (input_dict keys) ---
param_overrides = {
    # "gas_type": "He",
    # "nx": 62,
    # "V_bank": 100.0,
    # "S_gp": 1.0e21,
    # "sigma_in_cm2": 5.0e-15,        # ion-neutral cross section
    # "b_ion_neutral_drag": 1.0,      # ion-neutral drag scale
    # "end_surface_area_scale": 1.8,  # end-wall neutralization area
}

# --- Flag overrides (input_flags keys) ---
flag_overrides = {
    # "implicit_heat_conduction": True,
    # "ion_neutral_drag": True,
    # "ion_neutral_thermalization": False,
}

# --- Run controls (None => config default) ---
t_end = None            # [s] final time
dt = None               # [s] fixed step; None => adaptive
operator_split = None   # None => use implicit_heat_conduction flag
max_steps = None        # accepted-step cap; 0 => unlimited

output_path = "sim1d_run.h5"

## Run the simulation

In [ ]:
params, flags = default_config()
params.update(param_overrides)
flags.update(flag_overrides)

sim = LAPDSim1D(params, flags)
sim.start_simulation(
    t_end=t_end,
    dt=dt,
    operator_split=operator_split,
    max_steps=max_steps,
    progress_tracker=ProgressPrinter1D(),
)
result = sim.get_results()

out = Path(output_path)
out.parent.mkdir(parents=True, exist_ok=True)
sim.save_result(out, result, params=params, flags=flags)

s = summarize_result(result)
print(f"steps={result.steps}, final_time={result.final_time:.4e} s, saves={len(result.time)}, output={out}")
print(
    f"finite={s.finite}, "
    f"n=[{s.n_min:.3e}, {s.n_max:.3e}] cm^-3, "
    f"Te=[{s.Te_min:.3e}, {s.Te_max:.3e}] eV, "
    f"Ti=[{s.Ti_min:.3e}, {s.Ti_max:.3e}] eV"
)

## Plots

Port markers are the vertical dashed gray lines used by `bapsf_app`, at the LAPD probe ports (sim convention, z=0 at the source end):

| port | z [cm] |
|------|--------|
| 20   | 758    |
| 29   | 1045   |
| 40   | 1397   |

Edit `PORT_Z_SIM_CM` if your geometry differs.

In [ ]:
# Port probe positions [cm], sim convention (z=0 at source end).
PORT_Z_SIM_CM = {20: 758.0, 29: 1045.0, 40: 1397.0}


def add_port_lines(ax):
    """Draw the bapsf_app vertical dashed port markers on a z-axis plot."""
    for z in PORT_Z_SIM_CM.values():
        ax.axvline(z, color="gray", lw=0.8, ls="--", alpha=0.7)


# Reproduce the CLI time axis (t=0 at main discharge, milliseconds).
time_origin = psr._time_origin(result, "main_discharge")
time_scale, time_label = psr._time_unit("ms")
shifted_s = np.asarray(result.time, dtype=float) - time_origin
t_plot = shifted_s * time_scale
t_slice_ms = shifted_s * 1.0e3
z_cm = np.asarray(result.z_cm, dtype=float)
phase_events = psr._shifted_phase_events(result, time_origin, time_scale)

In [ ]:
figures = {
    "Summary": psr._plot_summary(result, t_plot, time_label, phase_events),
    "Densities": psr._plot_densities(result, z_cm, t_plot, time_label, phase_events),
    "Temperatures": psr._plot_temperatures(result, z_cm, t_plot, time_label, phase_events),
    "Velocity": psr._plot_velocity(result, z_cm, t_plot, time_label, phase_events),
    "Energy terms": psr._plot_energy_terms(result, t_plot, time_label, phase_events),
    "Cathode": psr._plot_cathode(result, t_plot, time_label, phase_events),
    "Phase": psr._plot_phase(result, t_plot, time_label, phase_events),
}

for name, fig in figures.items():
    if fig is None:
        continue
    # Add port markers only to panels whose x-axis is z [cm] (skips time-axis and colorbar axes).
    for ax in fig.axes:
        if ax.get_xlabel().startswith("z"):
            add_port_lines(ax)
    display(fig)
    plt.close(fig)

## Time-slice profiles at 15 ms and 19 ms

Main-discharge-relative times. Each figure snaps to the nearest saved timestep; the title reports the actual time used. Make sure the run reaches ~19 ms of main discharge, or edit `SLICE_TIMES_MS`.

In [ ]:
SLICE_TIMES_MS = (15.0, 19.0)

for slice_ms in SLICE_TIMES_MS:
    fig = psr._plot_time_slice_summary(
        result=result,
        z_cm=z_cm,
        t_ms=t_slice_ms,
        slice_time_ms=slice_ms,
    )
    for ax in fig.axes:  # all four panels share a z [cm] x-axis
        add_port_lines(ax)
    display(fig)
    plt.close(fig)